# Qwen3-1.7B DPO (v4) — Colab Inference Notebook

Self-contained notebook for testing the **DPO-trained Qwen3-1.7B** model against the base `Qwen/Qwen3-1.7B` on math reasoning.

## ⚠️ Known model accuracy

This is a **research prototype on a 1.7B model** — do not expect strong results.

| Condition | GSM8K accuracy (full 1319-question eval) | Avg tokens |
|-----------|------------------------------------------|------------|
| Causal SFT baseline | 77.3% | 169 |
| **DPO v4 (this model)** | **75.1%** | **172** |

DPO did not improve accuracy over the SFT baseline on this run — the causal training data (PNS-pruned chains) is very compressed, so the preference signal is weak. The token reduction compared to the **base model** is the main observable effect at this scale.

## What this notebook tests

- **Base Qwen3-1.7B** — no fine-tuning, verbose, lower accuracy  
- **DPO v4** — SFT + DPO merged into base, trained to prefer causally compressed correct chains

## Loading strategy

1. `gdown` downloads the model folder from Google Drive into `/content/dpo_model/`.
2. **DPO model** (fully merged, no adapter) loaded in **4-bit** — ~0.9 GB VRAM.
3. **Base Qwen3-1.7B** loaded separately from HuggingFace in 4-bit for comparison.
4. Both run on **10 embedded test problems** (9 GSM8K + 1 MATH-500, hardcoded — no upload needed).

## Runtime

**Runtime → Change runtime type → T4 GPU**. Two 4-bit Qwen3-1.7B instances take ~1.8 GB total — well within T4's 16 GB.

## 1 · Install dependencies

`bitsandbytes` for 4-bit quantization, `accelerate` for `device_map="auto"`, `gdown` for Drive download.

In [ ]:
!pip install -q --upgrade transformers accelerate bitsandbytes sentencepiece gdown

## 2 · Download DPO model from public Google Drive

`gdown` fetches the entire public folder into `/content/dpo_model/`.

**What to upload to Drive**: the folder `models/dpo/qwen3_causal_dpo_v4/` from the project repo  
(files needed: `model.safetensors`, `config.json`, `generation_config.json`, `tokenizer.json`, `tokenizer_config.json`, `chat_template.jinja`).  
Do **not** include the `checkpoint-100` / `checkpoint-192` subdirectories — those are intermediate LoRA-only checkpoints and not needed here.

After uploading, paste the Drive **folder share link** below and extract the folder ID  
(the long alphanumeric string after `/folders/` in the URL).

In [ ]:
import os, json, glob
import gdown

# Public Google Drive folder — contains model.safetensors, config.json,
# tokenizer files, chat_template.jinja, and optionally sample_test.jsonl.
FOLDER_ID     = "1MoquHqcdyR4nKo0b7Ufd1l5lFQ0nypg-"
DOWNLOAD_ROOT = "/content/dpo_model"

print(f"Downloading DPO model from Drive folder: {FOLDER_ID}")
os.makedirs(DOWNLOAD_ROOT, exist_ok=True)
gdown.download_folder(
    f"https://drive.google.com/drive/folders/{FOLDER_ID}",
    output=DOWNLOAD_ROOT,
    quiet=False,
    use_cookies=False,
)

# Locate the directory that has config.json (the root of the merged model).
# We look for config.json — NOT adapter_config.json — because this is a
# full merged model, not a LoRA adapter.
hits = glob.glob(os.path.join(DOWNLOAD_ROOT, "**", "config.json"), recursive=True)
# Exclude checkpoint subdirectories (those contain adapter_config, not config)
root_hits = [
    h for h in hits
    if "checkpoint-" not in h
    and not os.path.exists(os.path.join(os.path.dirname(h), "adapter_config.json"))
]
assert root_hits, (
    f"config.json not found under {DOWNLOAD_ROOT}.\n"
    f"Make sure you uploaded the root model folder (not just checkpoints).\n"
    f"Tree: {[(r, f) for r, _, files in os.walk(DOWNLOAD_ROOT) for f in files]}"
)
MODEL_PATH = os.path.dirname(root_hits[0])

# Locate chat_template.jinja
tmpl_hits = glob.glob(os.path.join(MODEL_PATH, "chat_template.jinja"))
HAS_CHAT_TEMPLATE = bool(tmpl_hits)

# Locate sample_test.jsonl if user included one
sample_hits      = glob.glob(os.path.join(DOWNLOAD_ROOT, "**", "sample_test.jsonl"), recursive=True)
SAMPLE_DATA_PATH = sample_hits[0] if sample_hits else None

# Sanity-check required files
for name in ("config.json", "model.safetensors"):
    path = os.path.join(MODEL_PATH, name)
    assert os.path.exists(path), f"Missing '{name}' in model folder: {MODEL_PATH}"

print(f"\nModel dir    : {MODEL_PATH}")
print(f"Contents     : {sorted(os.listdir(MODEL_PATH))}")
print(f"Chat template: {'found' if HAS_CHAT_TEMPLATE else 'not found — will use tokenizer default'}")
print(f"Sample data  : {SAMPLE_DATA_PATH or '(not in Drive — hardcoded samples will be used)'}")

## 3 · Read `config.json` (and `chat_template.jinja` if present)

In [ ]:
with open(os.path.join(MODEL_PATH, "config.json"), encoding="utf-8") as f:
    MODEL_CFG = json.load(f)

CHAT_TEMPLATE = None
if HAS_CHAT_TEMPLATE:
    with open(os.path.join(MODEL_PATH, "chat_template.jinja"), encoding="utf-8") as f:
        CHAT_TEMPLATE = f.read()

print("Model config:")
for k in ("model_type", "architectures", "hidden_size", "num_hidden_layers",
          "num_attention_heads", "vocab_size"):
    if k in MODEL_CFG:
        print(f"  {k}: {MODEL_CFG[k]}")

print(f"\nThis is a FULLY MERGED model (SFT + DPO weights combined into base Qwen3-1.7B).")
print(f"No PEFT adapter needed — load directly with AutoModelForCausalLM.")
if CHAT_TEMPLATE:
    print(f"\nchat_template.jinja: {len(CHAT_TEMPLATE)} chars")

## 4 · Imports & 4-bit quantization config

NF4 with bf16 compute — standard QLoRA-style inference. Qwen3-1.7B in 4-bit fits in ~1 GB VRAM.

In [ ]:
import re, time, gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

assert torch.cuda.is_available(), "4-bit loading via bitsandbytes requires a CUDA GPU. Switch runtime to GPU."

COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

BNB_CONFIG = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_use_double_quant = True,
    bnb_4bit_compute_dtype    = COMPUTE_DTYPE,
)

# Capped at 512 for the test run — the 5 easy examples need at most ~150 tokens.
# Keeping it low prevents Colab from OOMing on unexpectedly long generations.
MAX_NEW_TOKENS = {"gsm8k": 512, "math500": 1024, "external": 512}

BASE_MODEL_NAME = "Qwen/Qwen3-1.7B"

print(f"GPU: {torch.cuda.get_device_name(0)} | compute dtype: {COMPUTE_DTYPE}")
print(f"Base model for comparison: {BASE_MODEL_NAME}")
print(f"max_new_tokens cap: {MAX_NEW_TOKENS}")

## 5 · Answer-extraction helpers

Copied from `algo/equivalent_ans.py`. Local-only grading — no LLM judge needed.

In [ ]:
def _extract_boxed(text: str) -> str:
    """Last \\boxed{...}; handles nested braces."""
    results, start = [], 0
    while True:
        idx = text.find(r"\boxed{", start)
        if idx == -1:
            break
        depth = 0
        for i in range(idx + 7, len(text)):
            if text[i] == "{":
                depth += 1
            elif text[i] == "}":
                if depth == 0:
                    results.append(text[idx + 7:i])
                    start = i + 1
                    break
                depth -= 1
        else:
            break
    return results[-1].strip() if results else ""


def _normalize(text: str) -> str:
    t = text.strip()
    t = re.sub(r"\\left|\\right|\\,|\\!", "", t)
    t = re.sub(r"\^\\circ|\\circ|\\degree|°", "", t)
    t = re.sub(r"\\text\{([^}]*)\}", r"\1", t)
    t = re.sub(r"\\dfrac", r"\\frac", t)
    t = re.sub(r"\\tfrac", r"\\frac", t)
    t = re.sub(r"\$+", "", t)
    t = re.sub(r"\s+", "", t)
    return t.lower()


def is_correct_local(extracted: str, ground_truth: str) -> bool:
    if not extracted:
        return False
    if _normalize(extracted) == _normalize(ground_truth):
        return True
    try:
        return float(extracted.replace(",", "").strip()) == float(str(ground_truth).replace(",", "").strip())
    except (ValueError, TypeError):
        return False


def extract_answer(text: str, dataset: str) -> str:
    boxed = _extract_boxed(text)
    if boxed:
        return boxed
    if dataset == "gsm8k":
        m = re.search(r"####\s*([\d,.\-]+)", text)
        if m:
            return m.group(1).replace(",", "").strip()
    return ""


def strip_think(text: str) -> str:
    """Return post-</think> content. Falls back to full text so extract_answer
    can still find \\boxed{} inside the reasoning chain."""
    return text.split("</think>", 1)[1].strip() if "</think>" in text else text


def count_steps(text: str) -> int:
    """Count double-newline-separated paragraphs inside the <think> block."""
    if "<think>" in text and "</think>" in text:
        think = text.split("<think>", 1)[1].split("</think>", 1)[0]
        return len([p for p in re.split(r"\n\n+", think) if p.strip()])
    return len([p for p in re.split(r"\n\n+", text) if p.strip()])


print("Helpers loaded.")

## 6 · Tokenizer + prompt builder

Loaded from the model folder (saved there by `train_dpo.py → tokenizer.save_pretrained()`).  
If `chat_template.jinja` was included in the Drive upload, it overwrites the tokenizer's default template  
to guarantee we use the exact template from training.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # left-pad for generation — matches eval_correct.py

if CHAT_TEMPLATE:
    tokenizer.chat_template = CHAT_TEMPLATE
    print("Chat template overridden from chat_template.jinja")


def build_prompt(tok, question: str) -> str:
    """Mirrors build_prompt() in sft/eval_correct.py and dpo/train_dpo.py."""
    msgs   = [{"role": "user", "content": question.strip()}]
    kwargs = dict(tokenize=False, add_generation_prompt=True)
    try:
        return tok.apply_chat_template(msgs, enable_thinking=True, **kwargs)
    except TypeError:
        return tok.apply_chat_template(msgs, **kwargs)


# Sanity check
_sample = build_prompt(tokenizer, "What is 2 + 2?")
print("-- sample prompt --")
print(_sample[:300])
print(f"\nprompt length: {len(_sample)} chars")

## 7 · Load DPO model (4-bit, from Drive)

This is a **fully merged model** — SFT weights and DPO LoRA were both merged into `Qwen/Qwen3-1.7B` and saved  
as a single `model.safetensors` file by `train_dpo.py`. Load it exactly like any HuggingFace model.

In [ ]:
print(f"Loading DPO model from: {MODEL_PATH}")
print("(SFT + DPO LoRA fully merged into Qwen3-1.7B base — no PEFT adapter needed)")

dpo_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=BNB_CONFIG,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager",
)
dpo_model.eval()
print(f"  VRAM after DPO model load: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

## 8 · Load base Qwen3-1.7B (4-bit, from HuggingFace)

The unmodified base model — no SFT, no DPO. Used as the comparison baseline.  
This mirrors the `with adapter_model.disable_adapter():` comparison in the Self-Distill notebook,  
but since the DPO model is fully merged (not an adapter), we load the base separately.

In [ ]:
print(f"Loading base model from HuggingFace: {BASE_MODEL_NAME}")

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    quantization_config=BNB_CONFIG,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager",
)
base_model.eval()
print(f"  VRAM after base model load: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

# Base tokenizer (same vocab as DPO but loaded from HF directly)
base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
if base_tokenizer.pad_token is None:
    base_tokenizer.pad_token = base_tokenizer.eos_token
base_tokenizer.padding_side = "left"
print("Base tokenizer loaded.")

## 9 · Test examples

**10 questions — 9 GSM8K + 1 MATH-500** — covering single-step, multi-step, percentage-based, and number theory problems. All questions have verified ground-truth answers.

These are sanity-check problems, not a rigorous accuracy benchmark. Given the known 75–77% accuracy on the full 1319-question eval, expect 7–8 out of 10 correct for the DPO model.

In [ ]:
_HARDCODED_SAMPLES = [
    # 1 — single multiplication
    {"dataset": "gsm8k", "answer": "540",
     "question": "James decides to run 3 sprints 3 times a week. "
                 "He runs 60 meters each sprint. "
                 "How many total meters does he run a week?"},
    # 2 — subtract then multiply (2 steps)
    {"dataset": "gsm8k", "answer": "18",
     "question": "Janet's ducks lay 16 eggs per day. She eats three for breakfast "
                 "every morning and bakes muffins for her friends every day with four. "
                 "She sells the remainder at the farmers' market daily for $2 per fresh "
                 "duck egg. How much in dollars does she make every day at the farmers' market?"},
    # 3 — rate × fraction of hour (2 steps)
    {"dataset": "gsm8k", "answer": "10",
     "question": "Weng earns $12 an hour for babysitting. "
                 "Yesterday, she just did 50 minutes of babysitting. "
                 "How much did she earn?"},
    # 4 — pages left after two days, halve (2 steps)
    {"dataset": "gsm8k", "answer": "42",
     "question": "Julie is reading a 120-page book. Yesterday, she was able to read "
                 "12 pages and today, she read twice as many pages as yesterday. "
                 "If she wants to read half of the remaining pages tomorrow, "
                 "how many pages should she read tomorrow?"},
    # 5 — percentage down payment (2 steps)
    {"dataset": "gsm8k", "answer": "56000",
     "question": "Roger bought a house for $100,000. He was able to pay 20% down, "
                 "and his parents paid off an additional 30% of the remaining balance. "
                 "How much money does Roger still owe on his house?"},
    # 6 — multi-source money problem (3 steps)
    {"dataset": "gsm8k", "answer": "5",
     "question": "Betty is saving money for a new wallet which costs $100. "
                 "Betty has only half of the money she needs. Her parents decided to give "
                 "her $15 for that purpose, and her grandparents twice as much as her parents. "
                 "How much more money does Betty need to buy the wallet?"},
    # 7 — percentage flowers (3 steps)
    {"dataset": "gsm8k", "answer": "35",
     "question": "Mark has a garden with flowers. He planted plants of three different "
                 "colors in it. Ten of them are yellow, and there are 80% more of those "
                 "in purple. There are only 25% as many green flowers as there are yellow "
                 "and purple flowers. How many flowers does Mark have in his garden?"},
    # 8 — tiered pricing (3 steps)
    {"dataset": "gsm8k", "answer": "64",
     "question": "Kylar went to the store to buy glasses for his new apartment. "
                 "One glass costs $5, but every second glass costs only 60% of the price. "
                 "Kylar wants to buy 16 glasses. How much does he need to pay for them?"},
    # 9 — profit calculation (4 steps)
    {"dataset": "gsm8k", "answer": "25",
     "question": "Sam bought a dozen boxes, each with 30 highlighter pens inside, "
                 "for $10 each box. He rearranged five of these boxes into packages of "
                 "six highlighters each and sold them for $3 per package. He sold the rest "
                 "of the highlighters separately at the rate of three pens per dollar. "
                 "How much profit did he make in total, in dollars?"},
    # 10 — MATH-500: number theory
    {"dataset": "math500", "answer": "9",
     "question": "How many positive whole-number divisors does 196 have?"},
]

if SAMPLE_DATA_PATH:
    with open(SAMPLE_DATA_PATH, encoding="utf-8") as f:
        SAMPLES = [json.loads(line) for line in f if line.strip()]
    print(f"Loaded {len(SAMPLES)} samples from {SAMPLE_DATA_PATH}")
else:
    SAMPLES = _HARDCODED_SAMPLES
    print(f"Using {len(SAMPLES)} hardcoded samples (no sample_test.jsonl in Drive folder)")

print(f"\nNote: this is a quick sanity check only — the model's known full-eval accuracy")
print(f"is 75.1% (DPO v4) vs 77.3% (SFT baseline) on 1319 GSM8K questions.\n")
for i, s in enumerate(SAMPLES, 1):
    print(f"  [{i:02d}] {s['dataset']:7s}  ans={s['answer']!r:8s}  {s['question'][:70]!r}")

## 10 · Generation + display helpers

In [ ]:
@torch.no_grad()
def generate_one(model, tok, question: str, dataset: str) -> dict:
    prompt = build_prompt(tok, question)
    enc = tok(prompt, return_tensors="pt", truncation=True, max_length=1024)
    enc = {k: v.to(model.device) for k, v in enc.items()}

    max_new = MAX_NEW_TOKENS.get(dataset, MAX_NEW_TOKENS["external"])
    t0 = time.time()
    out = model.generate(
        **enc,
        max_new_tokens=max_new,
        do_sample=False,
        temperature=1.0,
        top_p=1.0,
        pad_token_id=tok.eos_token_id,
    )
    in_len    = enc["input_ids"].shape[1]
    new_toks  = out[0][in_len:]
    response  = tok.decode(new_toks, skip_special_tokens=True)

    return {
        "prompt":      prompt,
        "response":    response,
        "extracted":   extract_answer(response, dataset),
        "answer_tail": strip_think(response),
        "new_tokens":  int(len(new_toks)),
        "step_count":  count_steps(response),
        "seconds":     time.time() - t0,
    }


def show_result(idx, ex, result, model_label):
    sep = "=" * 78
    print(sep)
    print(f"#{idx:02d}  [{ex.get('dataset','external')}]  model: {model_label}")
    print(sep)
    print("QUESTION:")
    q = ex["question"]
    print(q[:600] + ("..." if len(q) > 600 else ""))
    print()
    expected = ex.get("answer", "")
    print(f"EXPECTED ANSWER : {expected!r}")
    print(f"MODEL EXTRACTED : {result['extracted']!r}")
    if expected:
        print(f"CORRECT         : {is_correct_local(result['extracted'], expected)}")
    print(f"TOKENS / STEPS  : {result['new_tokens']} tok / {result['step_count']} steps / {result['seconds']:.1f}s")
    print("-" * 78)
    print("MODEL OUTPUT (post-</think> tail):")
    print(result["answer_tail"][:400] or "(empty — </think> never closed)")
    print("-- full response excerpt --")
    print(result["response"][:800].rstrip() + ("..." if len(result["response"]) > 800 else ""))
    print()


print("Generation helpers loaded.")

## 11 · Run DPO model on test examples

In [ ]:
dpo_results = []
for i, ex in enumerate(SAMPLES, 1):
    r = generate_one(dpo_model, tokenizer, ex["question"], ex.get("dataset", "external"))
    dpo_results.append((ex, r))
    show_result(i, ex, r, "DPO (Qwen3-1.7B, SFT+DPO merged)")

dpo_correct = sum(is_correct_local(r["extracted"], e["answer"]) for e, r in dpo_results if e.get("answer"))
dpo_graded  = sum(1 for e, _ in dpo_results if e.get("answer"))
print(f"\nDPO model accuracy on {len(SAMPLES)} examples: {dpo_correct}/{dpo_graded} = {dpo_correct/max(dpo_graded,1)*100:.1f}%")

## 12 · Run base Qwen3-1.7B (no fine-tuning) on same examples

In [ ]:
base_results = []
for i, ex in enumerate(SAMPLES, 1):
    r = generate_one(base_model, base_tokenizer, ex["question"], ex.get("dataset", "external"))
    base_results.append((ex, r))
    show_result(i, ex, r, "BASE (Qwen3-1.7B, no fine-tuning)")

base_correct = sum(is_correct_local(r["extracted"], e["answer"]) for e, r in base_results if e.get("answer"))
base_graded  = sum(1 for e, _ in base_results if e.get("answer"))
print(f"\nBase model accuracy on {len(SAMPLES)} examples: {base_correct}/{base_graded} = {base_correct/max(base_graded,1)*100:.1f}%")

## 13 · Comparison table: Base vs DPO

In [ ]:
row_fmt = "{:<3} {:<8} {:<20} {:<20} {:<20} {:<7} {:<7} {:<7} {:<7} {:<6}"
print(row_fmt.format("#", "dataset", "expected", "base", "dpo", "b_ok", "d_ok", "b_tok", "d_tok", "b_stp"))
print("-" * 110)
tok_base = tok_dpo = step_base = step_dpo = 0
for i, ((eb, rb), (ed, rd)) in enumerate(zip(base_results, dpo_results), 1):
    bok = is_correct_local(rb["extracted"], eb["answer"]) if eb.get("answer") else False
    dok = is_correct_local(rd["extracted"], ed["answer"]) if ed.get("answer") else False
    tok_base  += rb["new_tokens"]
    tok_dpo   += rd["new_tokens"]
    step_base += rb["step_count"]
    step_dpo  += rd["step_count"]
    print(row_fmt.format(
        i,
        eb.get("dataset", ""),
        str(eb.get("answer") or "")[:19],
        (rb["extracted"] or "")[:19],
        (rd["extracted"] or "")[:19],
        "Y" if bok else "N",
        "Y" if dok else "N",
        rb["new_tokens"],
        rd["new_tokens"],
        rb["step_count"],
    ))

n = max(len(SAMPLES), 1)
print("-" * 110)
print(f"BASE  accuracy: {base_correct}/{base_graded} = {base_correct/max(base_graded,1)*100:.1f}%  "
      f"avg tokens: {tok_base/n:.0f}  avg steps: {step_base/n:.1f}")
print(f"DPO   accuracy: {dpo_correct}/{dpo_graded} = {dpo_correct/max(dpo_graded,1)*100:.1f}%  "
      f"avg tokens: {tok_dpo/n:.0f}  avg steps: {step_dpo/n:.1f}")
print(f"\nDELTA : {(dpo_correct-base_correct)/n*100:+.1f} pp accuracy  "
      f"{(tok_dpo-tok_base)/n:+.0f} tokens avg  {(step_dpo-step_base)/n:+.1f} steps avg")
print()
print("Note: DPO was trained on causal (PNS-pruned) chains — lower token count indicates")
print("the model learned concise causal reasoning instead of verbose step-by-step chains.")

## 14 · External test data (CSV / JSONL)

If the instructor uploads their own test file, point `EXTERNAL_PATH` at it and run this cell.

**Supported schemas:**
- **JSONL** — one JSON per line. Keys: `question` (or `problem` / `prompt`), `answer` (or `ground_truth` / `answerKey`), optionally `dataset`.
- **CSV** — same column names. If no `answer` column, results print without correctness flag.

Column aliases match `sft/eval_correct.py` so the same test files work everywhere.

In [ ]:
QUESTION_KEYS = ("question", "problem", "prompt")
ANSWER_KEYS   = ("answer", "ground_truth", "answerKey")


def _pick(record, keys):
    for k in keys:
        if k in record and record[k] not in (None, ""):
            return record[k]
    return None


def load_external(path):
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    if path.lower().endswith(".jsonl"):
        with open(path, encoding="utf-8") as f:
            rows = [json.loads(line) for line in f if line.strip()]
    elif path.lower().endswith(".csv"):
        import csv
        with open(path, encoding="utf-8", newline="") as f:
            rows = list(csv.DictReader(f))
    else:
        raise ValueError(f"Unsupported extension: {path} (need .jsonl or .csv)")

    examples = []
    for r in rows:
        q = _pick(r, QUESTION_KEYS)
        if not q:
            continue
        examples.append({
            "question": str(q),
            "answer":   str(_pick(r, ANSWER_KEYS) or ""),
            "dataset":  str(r.get("dataset", "external")).lower(),
        })
    return examples


def run_external(path, model, tok, model_label="DPO", limit=None):
    examples = load_external(path)
    if limit:
        examples = examples[:limit]
    print(f"Loaded {len(examples)} examples from {path}")
    out = []
    correct = n_graded = 0
    for i, ex in enumerate(examples, 1):
        r = generate_one(model, tok, ex["question"], ex["dataset"])
        show_result(i, ex, r, model_label)
        out.append({**ex, **r})
        if ex["answer"]:
            n_graded += 1
            correct  += int(is_correct_local(r["extracted"], ex["answer"]))
    if n_graded:
        print(f"\nAccuracy: {correct}/{n_graded} = {correct/n_graded*100:.1f}%")
    else:
        print("\nNo ground-truth answers in file — accuracy not computed.")
    return out


# Uncomment and set the path to run on external data:
# EXTERNAL_PATH = "/content/drive/MyDrive/instructor_test.jsonl"   # or .csv
# external_outputs = run_external(EXTERNAL_PATH, dpo_model, tokenizer, model_label="DPO", limit=20)

### Optional — save all results to a JSONL file and download

In [ ]:
# OUTPUT_PATH = "dpo_inference_results.jsonl"
# with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
#     for (ex, rb), (_, rd) in zip(base_results, dpo_results):
#         f.write(json.dumps({
#             "question":             ex["question"],
#             "ground_truth":         ex.get("answer", ""),
#             "dataset":              ex.get("dataset", "gsm8k"),
#             "base_extracted":       rb["extracted"],
#             "dpo_extracted":        rd["extracted"],
#             "base_correct":         is_correct_local(rb["extracted"], ex.get("answer", "")),
#             "dpo_correct":          is_correct_local(rd["extracted"], ex.get("answer", "")),
#             "base_tokens":          rb["new_tokens"],
#             "dpo_tokens":           rd["new_tokens"],
#             "base_steps":           rb["step_count"],
#             "dpo_steps":            rd["step_count"],
#             "base_response":        rb["response"],
#             "dpo_response":         rd["response"],
#         }, ensure_ascii=False) + "\n")
# from google.colab import files
# files.download(OUTPUT_PATH)